# Робота з індексом стану рослинності

Тут реалізовані необхідні дії з даними, які було з сайту наукового центру NESDIS STAR

Створити віртуальне середовище (venv) в якому будуть встановлені всі необхідні бібліотеки та налаштування для даної лабораторної роботи

Ініціалізуємо необхідні бібліотеки, змінні та словарі

In [1]:
from urllib.request import urlopen
import time
import os
import re
import glob
import pandas as pd
import numpy as np

parsed_url = "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2026&type=Mean"
province_ids = list(range(1,28))
province_names = {
    1: "Cherkasy",
    2: "Chernihiv",
    3: "Chernivtsi",
    4: "Crimea",
    5: "Dnipropetrovs'k",
    6: "Donets'k",
    7: "Ivano-Frankivs'k",
    8: "Kharkiv",
    9: "Kherson",
    10: "Khmel'nyts'kyy",
    11: "Kiev",
    12: "Kiev City",
    13: "Kirovohrad",
    14: "Luhans'k",
    15: "L'viv",
    16: "Mykolayiv",
    17: "Odessa",
    18: "Poltava",
    19: "Rivne",
    20: "Sevastopol'",
    21: "Sumy",
    22: "Ternopil'",
    23: "Transcarpathia",
    24: "Vinnytsya",
    25: "Volyn",
    26: "Zaporizhzhya",
    27: "Zhytomyr"
}
changer_dict = {
        1: 25,
        2: 27,
        3: 26,
        4: 1,
        5: 4,
        6: 5,
        7: 9,
        8: 22,
        9: 23,
        10: 24,
        11: 11,
        12: 10,
        13: 12,
        14: 13,
        15: 14,
        16: 15,
        17: 16,
        18: 17,
        19: 18,
        20: 19,
        21: 20,
        22: 21,
        23: 7,
        24: 2,
        25: 3,
        26: 8,
        27: 6
    }
download_dir = "./data"
os.makedirs(download_dir, exist_ok=True)
dfs = []

Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних.

In [4]:
def file_already_exists(province_id):
    pattern = os.path.join(download_dir,f"vhi_province_{province_id}_1981_2026_Mean_*.csv")
    files = glob.glob(pattern)
    return len(files) > 0

def download_file():
    for province_id in province_ids:
        if file_already_exists(province_id):
            continue

        url = parsed_url.format(province_id=province_id)

        current_time = time.strftime("%d-%m-%Y_%H-%M-%S")
        destination = os.path.join(download_dir,f"vhi_province_{province_id}_1981_2026_Mean_{current_time}.csv")

        response = urlopen(url)
        text = response.read().decode("utf-8")

        clean_text = re.sub(r"<.*?>", "", text)

        with open(destination, "w", encoding="utf-8") as f:
            f.write(clean_text)

download_file()

Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області.

In [3]:
def csv_to_df():
    files = glob.glob("data/*.csv")

    for file in files:
        df = pd.read_csv(
            file,
            skiprows=2,
            header=None,
            names=["year", "week", "SMN", "SMT", "VCI", "TCI", "VHI"],
            usecols=[0, 1, 2, 3, 4, 5, 6],
            skipinitialspace=True
        )

        df.drop(columns=["SMN", "SMT", "VCI", "TCI"], inplace=True)

        df.replace(-1, np.nan, inplace=True)
        df.dropna(inplace=True)

        province_id = int(re.search(r'vhi_province_(\d+)', file).group(1))

        df["province_id"] = province_id
        df["province_name"] = province_names[province_id]

        dfs.append(df)

csv_to_df()

Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька).

In [4]:
def change_indexes():
    for df in dfs:
        df["province_id"] = df["province_id"].map(changer_dict)

change_indexes()

Реалізувати процедури для формування вибірок:

Потрібно об'єднати dataframes у один

In [5]:
def merge_dfs():
    global df_all
    df_all = pd.concat(dfs, ignore_index=True)

merge_dfs()

Ряд VHI для області за вказаний рік

In [7]:
def filter_province_year(province_id, year):
    df_filter_py = df_all[(df_all["province_id"] == province_id) & (df_all["year"] == year)]
    print(df_filter_py)

year = int(input("Введіть рік для виборки: "))
province = int(input("Введіть індекс області для виборки: "))
filter_province_year(province, year)

      year  week    VHI  province_id province_name
6316  2018     1  42.44           10     Kiev City
6317  2018     2  43.68           10     Kiev City
6318  2018     3  47.06           10     Kiev City
6319  2018     4  52.09           10     Kiev City
6320  2018     5  53.49           10     Kiev City
6321  2018     6  52.33           10     Kiev City
6322  2018     7  50.23           10     Kiev City
6323  2018     8  48.50           10     Kiev City
6324  2018     9  47.36           10     Kiev City
6325  2018    10  47.47           10     Kiev City
6326  2018    11  47.43           10     Kiev City
6327  2018    12  46.94           10     Kiev City
6328  2018    13  46.46           10     Kiev City
6329  2018    14  44.48           10     Kiev City
6330  2018    15  43.47           10     Kiev City
6331  2018    16  43.34           10     Kiev City
6332  2018    17  44.23           10     Kiev City
6333  2018    18  45.51           10     Kiev City
6334  2018    19  44.31        

Ряд VHI за вказаний діапазон років для вказаних областей

In [8]:
def filter_provinces_years(provinces_ids, years):
    df_filter_py = df_all[(df_all["province_id"].isin(provinces_ids)) & (df_all["year"].isin(years))]
    print(df_filter_py)

years = list(map(int, input("Введіть роки для виборки у форматі (yyyy yyyy yyyy): ").split()))
provinces = list(map(int, input("Введіть індекси областей для виборки у форматі (ii ii ii): ").split()))
filter_provinces_years(provinces, years)

       year  week    VHI  province_id province_name
34162  1991     1  33.17            2     Vinnytsya
34163  1991     2  36.01            2     Vinnytsya
34164  1991     3  38.33            2     Vinnytsya
34165  1991     4  39.89            2     Vinnytsya
34166  1991     5  38.53            2     Vinnytsya
...     ...   ...    ...          ...           ...
49056  2018    48  47.58            1        Crimea
49057  2018    49  47.33            1        Crimea
49058  2018    50  48.35            1        Crimea
49059  2018    51  49.07            1        Crimea
49060  2018    52  49.17            1        Crimea

[468 rows x 5 columns]


Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани

In [9]:
def filter_stats_provinces_years(provinces_ids, years):
    df_filter_py = df_all[(df_all["province_id"].isin(provinces_ids)) & (df_all["year"].isin(years))]
    stats = df_filter_py.groupby("province_name")["VHI"].agg(["min", "max", "mean", "median"])
    print(stats)

years = list(map(int, input("Введіть роки для виборки у форматі (yyyy yyyy yyyy): ").split()))
provinces = list(map(int, input("Введіть індекси областей для виборки у форматі (ii ii ii): ").split()))
filter_stats_provinces_years(provinces, years)

                   min    max       mean  median
province_name                                   
Dnipropetrovs'k  28.61  65.53  43.393072   43.38
Vinnytsya        31.54  76.06  50.129216   48.45
Volyn            22.11  68.18  46.337582   48.26
